<a href="https://colab.research.google.com/github/Akhila-010/GenAI_Tasks/blob/main/Hybrid_RAG_Knowledge_Assistant.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install -q openai langchain langchain-community langchain-text-splitters \
faiss-cpu sentence-transformers \
pypdf gradio rank_bm25 tiktoken

In [ ]:
import os
import faiss
import gradio as gr
import numpy as np

from getpass import getpass
from openai import OpenAI

from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.document_loaders import PyPDFLoader, TextLoader

from sentence_transformers import SentenceTransformer
from rank_bm25 import BM25Okapi

In [ ]:
OPENAI_API_KEY = getpass("Enter OpenAI API Key: ")

client = OpenAI(api_key=OPENAI_API_KEY)

Enter OpenAI API Key: ··········


In [ ]:
from google.colab import files

uploaded = files.upload()

Saving An-Assessment-Study-of-Gait-Biometric-Recognition-Using-Machine-Learning.pdf to An-Assessment-Study-of-Gait-Biometric-Recognition-Using-Machine-Learning.pdf


In [ ]:
documents = []

for file_name in uploaded.keys():

    if file_name.endswith(".pdf"):
        loader = PyPDFLoader(file_name)
        docs = loader.load()

    elif file_name.endswith(".txt") or file_name.endswith(".md"):
        loader = TextLoader(file_name)
        docs = loader.load()

    else:
        continue

    documents.extend(docs)

print("Total Documents Loaded:", len(documents))

Total Documents Loaded: 737


In [ ]:
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=500,
    chunk_overlap=100
)

chunks = text_splitter.split_documents(documents)

print("Total Chunks:", len(chunks))

Total Chunks: 4038


In [ ]:
chunk_texts = [chunk.page_content for chunk in chunks]

print(chunk_texts[0][:500])

Advances in Intelligent Systems and Computing 1141
Aboul Ella Hassanien
Roheet Bhatnagar
Ashraf Darwish   Editors
Advanced 
Machine Learning 
Technologies and 
Applications
Proceedings of AMLTA 2020


In [ ]:
embedding_model = SentenceTransformer(
    'all-MiniLM-L6-v2'
)

embeddings = embedding_model.encode(
    chunk_texts,
    show_progress_bar=True
)

print("Embedding Shape:", embeddings.shape)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Batches:   0%|          | 0/127 [00:00<?, ?it/s]

Embedding Shape: (4038, 384)


In [ ]:
dimension = embeddings.shape[1]

index = faiss.IndexFlatL2(dimension)

index.add(np.array(embeddings))

print("FAISS Index Ready")

FAISS Index Ready


In [ ]:
tokenized_corpus = [
    text.split() for text in chunk_texts
]

bm25 = BM25Okapi(tokenized_corpus)

print("BM25 Ready")

BM25 Ready


In [ ]:
def dense_search(query, top_k=3):

    query_embedding = embedding_model.encode([query])

    distances, indices = index.search(
        np.array(query_embedding),
        top_k
    )

    results = []

    for idx in indices[0]:
        results.append(chunk_texts[idx])

    return results

In [ ]:
def bm25_search(query, top_k=3):

    tokenized_query = query.split()

    scores = bm25.get_scores(tokenized_query)

    top_indices = np.argsort(scores)[::-1][:top_k]

    results = []

    for idx in top_indices:
        results.append(chunk_texts[idx])

    return results

In [ ]:
def hybrid_search(query, top_k=3):

    dense_results = dense_search(query, top_k)

    bm25_results = bm25_search(query, top_k)

    combined = dense_results + bm25_results

    unique_results = list(dict.fromkeys(combined))

    return unique_results[:top_k]

In [ ]:
def create_context(query):

    retrieved_docs = hybrid_search(query)

    context = "\n\n".join(retrieved_docs)

    return context

In [ ]:
def ask_rag(query):

    context = create_context(query)

    prompt = f"""
You are a helpful AI assistant.

Answer the question using ONLY the provided context.

Context:
{context}

Question:
{query}
"""

    response = client.chat.completions.create(
        model="gpt-4.1-mini",

        messages=[
            {
                "role": "user",
                "content": prompt
            }
        ],

        temperature=0.2
    )

    answer = response.choices[0].message.content

    return answer

In [ ]:
query = "What is the document about?"

answer = ask_rag(query)

print(answer)

The document is about a model designed to recognize specific entities such as court name, date of judgment, petitioner, respondent, name of the judge, and acts from legal documents. This model helps lawyers by providing key information before viewing the entire document, thereby reducing the number of documents they need to review. It relates to the use of technology and software to assist legal practice by digitizing legal services and supporting activities like document handling and legal proceedings.


In [ ]:
query = "Explain the main topic"

print("DENSE SEARCH")
print("="*50)

dense = dense_search(query)

for i, text in enumerate(dense):
    print(f"\nResult {i+1}:\n")
    print(text[:300])

print("\n\nBM25 SEARCH")
print("="*50)

bm = bm25_search(query)

for i, text in enumerate(bm):
    print(f"\nResult {i+1}:\n")
    print(text[:300])

print("\n\nHYBRID SEARCH")
print("="*50)

hybrid = hybrid_search(query)

for i, text in enumerate(hybrid):
    print(f"\nResult {i+1}:\n")
    print(text[:300])

DENSE SEARCH

Result 1:

it aims to summarize and discovers salient content on the basis of this representation.
There are two approaches based on the representation: topic representation and
indicator representation. Topic representation approaches transform the text into an
intermediate representation and construes the to

Result 2:

corpus of unstructured data. This step consists of two sub-steps, namely.
Concept Extraction using Hierarchical Dirichlet Process Topic modeling is a
process to uncover the semantic structures of text. For topic modeling, Hierarchical
Dirichlet Process (HDP) algorithm is used [ 15]. HDP is an extens

Result 3:

things very nicely. subject knowledge is excellent little bit angry. good con-
ceptual clarity. his knowledge about programming is ex-cellent and he deliver
tiny to tiny concepts to students. sir is the best to teach computer program-
ming they know very minute and important things regarding subject


BM25 SEARCH

Result 1:

various methodologies 

In [ ]:
def chatbot(query):

    return ask_rag(query)

interface = gr.Interface(
    fn=chatbot,

    inputs=gr.Textbox(
        lines=2,
        placeholder="Ask your question..."
    ),

    outputs="text",

    title="RAG Knowledge Assistant",

    description="Upload documents and chat with them using RAG."
)

interface.launch(share=True)

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://7011fe4d3ddd94e61e.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
